# OSNet baseline on MEVID-SC (same clothes only)

Tests **the OSNet baseline** (OSNet x1_0 MSMT17, one mean template per clip, frozen decision rule) on the same-clothes dataset `Data/MEVID-SameClothes`.

**No GPU needed.** The OSNet baseline's OSNet clip templates for every MEVID test clip are already saved on Drive from the earlier baseline run (all frames, the OSNet baseline's exact loader). Each template depends only on its own clip, and every MEVID-SC clip is an identical copy of an original test clip, so cell 2 keeps only the same-clothes clips. It checks person, camera and frame count clip by clip.

| Cell | What it does |
|---|---|
| 1 | Mount Drive, find MEVID-SC and the saved tracker templates |
| 2 | Accuracy: mAP and Rank-1 on MEVID-SC |
| 3 | The OSNet baseline's frozen accept/refuse rule: precision, coverage, silent errors |
| 4 | Consistent ID across cameras, same frozen two-group protocol as CSCI |

About 3 minutes in total.

In [ ]:
# 0. FIND or REBUILD the OSNet baseline's OSNet templates (run once, before cell 1)
import os, glob, shutil, subprocess, sys, numpy as np
from google.colab import drive
drive.mount('/content/drive')

SD, MD = '/content/drive/Shareddrives', '/content/drive/MyDrive'
DEST = f'{SD}/RESEARCH/Data/MEVID-results/OSNet-baseline'
SC_DIR = f'{SD}/RESEARCH/Data/MEVID-SameClothes'
NAME = 'test_templates_osnet_msmt17_all.npz'
os.makedirs(DEST, exist_ok=True)

print('Searching all of Drive (1-3 minutes) ...')
hits = [h for root in [MD, SD]
        for h in glob.glob(f'{root}/**/{NAME}', recursive=True)
        if os.path.dirname(h) != DEST]
if os.path.exists(f'{DEST}/{NAME}'):
    print('Already in place:', DEST)
elif hits:
    src = os.path.dirname(hits[0])
    print('FOUND your saved tracker results in', src)
    for f in [NAME, 'FROZEN_tau.txt']:
        if os.path.exists(f'{src}/{f}'):
            shutil.copy(f'{src}/{f}', DEST)
    ft = glob.glob(f'{os.path.dirname(src)}/OSNet-finetuned/'
                   'test_templates_ft_all.npz')
    if ft:
        os.makedirs(f'{os.path.dirname(DEST)}/OSNet-finetuned',
                    exist_ok=True)
        shutil.copy(ft[0], f'{os.path.dirname(DEST)}/OSNet-finetuned/')
    print('Copied to', DEST)
else:
    print('Not found anywhere: rebuilding on MEVID-SC (A100, ~20-30 min)')
    import torch
    assert torch.cuda.is_available(), 'Switch the runtime to A100 GPU'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torchreid', 'gdown'], check=True)
    import torch.nn.functional as F, torchreid, gdown
    import torchvision.transforms as T
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm

    LOCAL = '/content/mevid_sc'
    if not os.path.isdir(f'{LOCAL}/bbox_test'):
        os.makedirs(LOCAL, exist_ok=True)
        print('Unpacking MEVID-SC test images (about 10 minutes) ...')
        subprocess.run(['tar', '-xf', f'{SC_DIR}/mevid-sc-bbox-test.tar',
                        '-C', LOCAL], check=True)
    A = f'{SC_DIR}/annotations'
    names = [l.strip() for l in open(f'{A}/test_name.txt') if l.strip()]
    info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
    orig = np.loadtxt(f'{A}/original_index_test.txt').astype(np.int64)
    n2p = {f: os.path.join(dp, f) for dp, _, fs in
           os.walk(f'{LOCAL}/bbox_test') for f in fs}

    # tracker's exact OSNet: x1_0, MSMT17 weights, 256x128, L2-normalised
    model = torchreid.models.build_model(name='osnet_x1_0',
                                         num_classes=1000, loss='softmax',
                                         pretrained=True)
    ckpt = os.path.expanduser('~/.cache/torch/checkpoints/'
                              'osnet_x1_0_msmt17.pth')
    os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for url in ['https://drive.google.com/uc?id=112EMUfBPYeYg70w-'
                'syK6V6Mx8-Qb9Q1M', 'https://drive.google.com/uc?id='
                '1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x']:
        if not os.path.exists(ckpt):
            try:
                gdown.download(url, ckpt, quiet=True, fuzzy=True)
            except Exception as e:
                print('download failed:', e)
    assert os.path.exists(ckpt), 'MSMT17 weights did not download'
    torchreid.utils.load_pretrained_weights(model, ckpt)
    model = model.cuda().eval()
    TF = T.Compose([T.Resize((256, 128)), T.ToTensor(),
                    T.Normalize([0.485, 0.456, 0.406],
                                [0.229, 0.224, 0.225])])
    items = [(n2p[os.path.basename(names[k])], t)
             for t, (s, e) in enumerate(info[:, :2])
             for k in range(s, e + 1)]

    class Frames(Dataset):
        def __len__(self): return len(items)
        def __getitem__(self, i):
            p, t = items[i]
            return TF(Image.open(p).convert('RGB')), t

    per = [[] for _ in range(len(info))]
    dl = DataLoader(Frames(), batch_size=512, pin_memory=True,
                    num_workers=min(12, os.cpu_count() or 2))
    with torch.no_grad():
        for x, t in tqdm(dl, desc=f'OSNet on {len(items):,} frames'):
            f = model(x.cuda(non_blocking=True))
            f = F.normalize(f[0] if isinstance(f, (tuple, list)) else f,
                            dim=1).cpu().numpy()
            for k, ti in enumerate(t.numpy()):
                per[ti].append(f[k].astype(np.float16))

    # same layout as the original file: 1,754 MEVID test rows,
    # filled for the MEVID-SC clips (cell 2 keeps only those)
    tm = np.zeros((1754, 512), np.float32)
    sm = np.ones(1754, np.float32); nf = np.zeros(1754, np.int64)
    for t in range(len(info)):
        Ft = np.stack(per[t]).astype(np.float32)
        m = Ft.mean(0); m /= max(np.linalg.norm(m), 1e-8)
        r = orig[t]
        tm[r], sm[r], nf[r] = m, float((Ft @ m).min()), len(Ft)
    np.savez(f'{DEST}/{NAME}', templates=tm, s_min=sm, n_frames=nf)
    if not os.path.exists(f'{DEST}/FROZEN_tau.txt'):
        with open(f'{DEST}/FROZEN_tau.txt', 'w') as f:
            f.write('tau=0.86\nsource=frozen on MEVID train '
                    '(registry v4)\n')
    print('Rebuilt and saved to', DEST)

In [ ]:
# 1. Mount Drive; find MEVID-SC and the saved tracker (OSNet) templates
import os, glob, json, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

SD, MD = '/content/drive/Shareddrives/*', '/content/drive/MyDrive'


def find(patterns, required=True):
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(patterns[0])
    return None


SC_DIR = os.path.dirname(find(
    [f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
     f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
ZS = 'test_templates_osnet_msmt17_all.npz'
F_ZS = find([f'{MD}/MEVID-results/OSNet-baseline/{ZS}',
             f'{SD}/Data/MEVID-results/OSNet-baseline/{ZS}',
             f'{SD}/Research/2026/MEVID/OSNet-baseline/{ZS}',
             f'{MD}/**/OSNet-baseline/{ZS}', f'{SD}/**/OSNet-baseline/{ZS}'])
OSN_DIR = os.path.dirname(F_ZS)
F_FT = find([f'{os.path.dirname(OSN_DIR)}/OSNet-finetuned/'
             'test_templates_ft_all.npz'], required=False)
F_CSCI = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes',
               f'{SD}/**/CSCI-SameClothes'], required=False)

# frozen identity-level tau from the registry notebook (v4), if saved
TAU = None
if os.path.exists(f'{OSN_DIR}/FROZEN_tau.txt'):
    for line in open(f'{OSN_DIR}/FROZEN_tau.txt'):
        if line.startswith('tau='):
            TAU = float(line.split('=')[1])
FLOOR, CAP, MARGIN = 0.65, 0.80, 0.05      # frozen OSNet baseline values

OUT = os.path.join(os.path.dirname(OSN_DIR), 'Tracker-SameClothes')
os.makedirs(OUT, exist_ok=True)
print('MEVID-SC           :', SC_DIR)
print('Tracker templates  :', F_ZS)
print('Fine-tuned OSNet   :', F_FT or 'not found (skipped)')
print('CSCI on MEVID-SC   :', F_CSCI or 'not found (no comparison)')
print('Frozen tau         :', TAU or 'not found (original rule only)')
print('Results go to      :', OUT)


In [ ]:
# 2. Tracker templates for MEVID-SC + accuracy (mAP, Rank-k)
A = f'{SC_DIR}/annotations'
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
orig = np.loadtxt(f'{A}/original_index_test.txt').astype(np.int64)
qidx = np.loadtxt(f'{A}/query_IDX.txt').astype(np.int64)
gidx = np.setdiff1d(np.arange(len(info)), qidx)
pid, oid, cam = info[:, 2], info[:, 3], info[:, 4]
frames = info[:, 1] - info[:, 0] + 1

models = {'Tracker (OSNet, zero-shot)': F_ZS}
if F_FT:
    models['Tracker (OSNet, fine-tuned)'] = F_FT
T, SMIN = {}, {}
for name, path in models.items():
    z = np.load(path)
    assert len(z['templates']) == 1754, f'{name}: not the MEVID test order'
    if 'n_frames' in z.files:
        # same clip <=> same number of frames, for every MEVID-SC clip
        assert (z['n_frames'][orig] == frames).all(), f'{name}: clip mismatch'
    T[name], SMIN[name] = z['templates'][orig], z['s_min'][orig]
assert (pd.Series(oid).groupby(pid).nunique() == 1).all()
print(f'Clip check PASSED: {len(info)} clips, {len(np.unique(pid))} people, '
      f'one outfit each; {len(qidx)} queries, {len(gidx)} gallery clips')


def ap_cmc(order, good, junk):
    order = order[~np.isin(order, junk)]
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc


def evaluate(Tm):
    S = Tm[qidx] @ Tm[gidx].T
    aps, cmcs = [], []
    for i, q in enumerate(qidx):
        sp = np.flatnonzero(pid[gidx] == pid[q])
        sc = np.flatnonzero(cam[gidx] == cam[q])
        good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
        if good.size:
            a, c = ap_cmc(np.argsort(-S[i]), good, junk)
            aps.append(a); cmcs.append(c[:20])
    cmc = np.mean(cmcs, 0)
    return {'queries': len(aps), 'mAP %': round(100 * np.mean(aps), 1),
            'Rank-1 %': round(100 * cmc[0], 1),
            'Rank-5 %': round(100 * cmc[4], 1),
            'Rank-10 %': round(100 * cmc[9], 1)}


acc = pd.DataFrame({n: evaluate(t) for n, t in T.items()}).T
if F_CSCI and os.path.exists(f'{F_CSCI}/accuracy_csci_sc.csv'):
    c = pd.read_csv(f'{F_CSCI}/accuracy_csci_sc.csv', index_col=0)
    acc.loc['CSCI (reference)'] = c.loc['MEVID-SC (same clothes)']
print('\nACCURACY on MEVID-SC (same clothes, cross-camera)')
display(acc)
acc.to_csv(f'{OUT}/accuracy_tracker_sc.csv')
np.savez(f'{OUT}/test_templates_tracker_sc.npz', pid=pid, oid=oid, cam=cam,
         qidx=qidx, gidx=gidx, orig_idx=orig,
         **{('zs' if 'zero' in n else 'ft'): t for n, t in T.items()},
         **{('zs' if 'zero' in n else 'ft') + '_smin': s
            for n, s in SMIN.items()})


In [ ]:
# 3. The OSNet baseline's FROZEN accept/refuse rule on MEVID-SC
#    accept the best other-camera match b only if
#      best_sim >= theta(b) = min(max(s_min[b], 0.65), 0.80)
#      and best_sim - runner_up >= 0.05
#    runner_up: 2nd-best clip (original rule) or the best clip NOT linked
#    to b (identity-level registry fix, frozen tau). No labels used.
def decide(Tm, smin, tau):
    S = Tm[qidx] @ Tm[gidx].T
    rows = []
    for i, q in enumerate(qidx):
        cand = np.flatnonzero(cam[gidx] != cam[q])
        if cand.size < 2 or not np.any(pid[gidx][cand] == pid[q]):
            continue
        s = S[i, cand]; o = np.argsort(-s)
        top = gidx[cand[o]]; sims = s[o]
        b = top[0]
        theta = min(max(float(smin[b]), FLOOR), CAP)
        link = Tm[top] @ Tm[b]
        free = np.flatnonzero(link[1:] < tau) + 1
        runner = float(sims[free[0]]) if free.size else -1.0
        rows.append({'query': int(q), 'match': int(b),
                     'best_sim': float(sims[0]), 'theta': theta,
                     'margin': float(sims[0]) - runner,
                     'correct': bool(pid[b] == pid[q]),
                     'accept': bool(sims[0] >= theta
                                    and sims[0] - runner >= MARGIN)})
    return pd.DataFrame(rows)


def summarise(d):
    a = d[d.accept]; wrong = (~d.correct).sum()
    return {'queries': len(d),
            'top-1 correct % (no rule)': round(100 * d.correct.mean(), 1),
            'accepted (coverage %)': round(100 * len(a) / len(d), 1),
            'precision of accepted %':
                round(100 * a.correct.mean(), 1) if len(a) else np.nan,
            'silent errors': int((~a.correct).sum()),
            'wrong matches caught %':
                round(100 * (wrong - (~a.correct).sum()) / max(1, wrong), 1)}


rules = {'original rule (2nd-best clip)': 1.01}
if TAU:
    rules[f'identity-level rule (frozen tau {TAU})'] = TAU
rows = {}
for n in T:
    for r, tau in rules.items():
        d = decide(T[n], SMIN[n], tau)
        d.to_csv(f'{OUT}/audit_{"zs" if "zero" in n else "ft"}_'
                 f'{"orig" if tau > 1 else "tau"}.csv', index=False)
        rows[f'{n}, {r}'] = summarise(d)
rule = pd.DataFrame(rows).T
print('TRACKER DECISION RULE on MEVID-SC (frozen, never tuned on test)')
display(rule)
rule.to_csv(f'{OUT}/decision_rule_tracker_sc.csv')


In [ ]:
# 4. CONSISTENT ID across cameras: one global ID per clip, scored.
#    Same frozen two-group protocol as CSCI (calibrate the linking
#    threshold on one group's people, freeze, evaluate on the other).
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment

P = f'{F_CSCI}/PROTOCOL_frozen_sc.json' if F_CSCI else None
if P and os.path.exists(P):
    fold_of = json.load(open(P))['folds']
    print('Using the frozen MEVID-SC person groups from', P)
else:
    raise FileNotFoundError('PROTOCOL_frozen_sc.json not found; run the '
                            'CSCI MEVID-SC notebook cell 3 first')
fold = np.array([fold_of[str(int(p))] for p in pid])
TARGETS = [0.95, 0.90]


def score_ids(pred, p, c):
    Pm = {x: i for i, x in enumerate(np.unique(p))}
    Cm = {x: j for j, x in enumerate(np.unique(pred))}
    M = np.zeros((len(Pm), len(Cm)))
    for a, b in zip(p, pred):
        M[Pm[a], Cm[b]] += 1
    r, cc = linear_sum_assignment(-M)
    sp = p[:, None] == p[None, :]; sc = pred[:, None] == pred[None, :]
    off = ~np.eye(len(p), dtype=bool); tp = (sp & sc & off).sum()
    xc = sp & (c[:, None] != c[None, :])
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': 100 * tp / max(1, (sc & off).sum()),
            'pair recall %': 100 * tp / max(1, (sp & off).sum()),
            'cross-camera recall %': 100 * (xc & sc).sum() / max(1, xc.sum())}


def tree(Tm, idx):
    d = 1 - np.clip(Tm[idx] @ Tm[idx].T, -1, 1)
    return linkage(d[np.triu_indices(len(idx), 1)], 'average')


SIMS = np.round(np.arange(0.10, 0.991, 0.01), 2)
rows = []
for n, Tm in T.items():
    for cal, ev in [('A', 'B'), ('B', 'A')]:
        ic, ie = np.flatnonzero(fold == cal), np.flatnonzero(fold == ev)
        Zc, Ze = tree(Tm, ic), tree(Tm, ie)
        sweep = pd.DataFrame([dict(s=s, **score_ids(
            fcluster(Zc, 1 - s, 'distance'), pid[ic], cam[ic]))
            for s in SIMS])
        pts = {'max IDF1': float(sweep.loc[sweep['IDF1 %'].idxmax(), 's'])}
        for tg in TARGETS:
            ok = sweep[sweep['pair precision %'] >= 100 * tg]
            pts[f'pair precision >= {int(100 * tg)}%'] = (
                float(ok['s'].min()) if len(ok) else float(SIMS.max()))
        for name, s in pts.items():
            pc = sweep.loc[sweep.s == s, 'pair precision %'].iloc[0]
            res = score_ids(fcluster(Ze, 1 - s, 'distance'),
                            pid[ie], cam[ie])
            rows.append({'model': n, 'calibrate->evaluate': f'{cal}->{ev}',
                         'operating point': name, 'link sim >=': s,
                         'predicted pair precision %': round(pc, 1),
                         **{k: round(v, 2) for k, v in res.items()}})
cid = pd.DataFrame(rows)
cid.to_csv(f'{OUT}/consistent_id_tracker_sc.csv', index=False)
print('CONSISTENT ID on MEVID-SC, tracker (held-out group)')
display(cid)

cols = ['IDF1 %', 'IDs per person', 'people per ID', 'pair precision %',
        'cross-camera recall %']
mean = cid.groupby(['model', 'operating point'])[cols].mean()
c = f'{F_CSCI}/step3_consistent_id_csci_sc.csv'
if os.path.exists(c):
    cs = pd.read_csv(c).groupby('operating point')[cols].mean()
    cs.index = pd.MultiIndex.from_product([['CSCI (reference)'], cs.index])
    mean = pd.concat([mean, cs])
print('\nMean of both directions (ideal: IDF1 100, 1 ID per person, '
      '1 person per ID)')
display(mean.round(1))
print('Saved all tracker results to', OUT)
